# Crypto Premium Index Dataset

Perpetual futures OHLCV and premium index data for funding rate arbitrage strategy.

| Property | Value |
|----------|-------|
| **Provider** | Binance Public API |
| **Asset Class** | Cryptocurrency |
| **Frequency** | 1h (OHLCV), 8h (premium) |
| **Symbols** | 20 perpetual futures |
| **Coverage** | 2020-2025 |
| **Size** | ~70 MB |
| **API Key** | None (free) |
| **Loader** | `load_crypto_perps()`, `load_crypto_premium()` |

In [1]:
"""Crypto Premium Index - download, explore, and update workflow."""

import json
from pathlib import Path

import polars as pl
import yaml

## 1. Configuration

The crypto universe is defined in `config.yaml`. Organized by market segment:
major cryptocurrencies, DeFi tokens, and Layer 1 blockchains.

In [2]:
# Load and display configuration
config_path = Path("config.yaml")
config = yaml.safe_load(config_path.read_text())
crypto_config = config["crypto"]

print("=== Crypto Configuration ===")
print(f"Provider: {crypto_config['provider']}")
print(f"Market: {crypto_config['market']}")
print(f"Date range: {crypto_config['start']} to {crypto_config['end']}")
print(f"Premium interval: {crypto_config['interval']}")
print("\nCategories:")
for category, info in crypto_config["symbols"].items():
    symbols = info["symbols"]
    print(f"  {category}: {len(symbols)} tokens - {info['description']}")
    print(f"    {', '.join(symbols[:5])}{'...' if len(symbols) > 5 else ''}")

total_symbols = sum(len(info["symbols"]) for info in crypto_config["symbols"].values())
print(f"\nTotal: {total_symbols} symbols")

=== Crypto Configuration ===
Provider: binance_bulk
Market: futures
Date range: 2020-01-01 to 2025-12-31
Premium interval: 8h

Categories:
  major: 9 tokens - Major perpetual futures for premium index analysis
    BTCUSDT, ETHUSDT, BNBUSDT, SOLUSDT, XRPUSDT...
  defi: 5 tokens - DeFi tokens with active perpetual markets
    LINKUSDT, UNIUSDT, AAVEUSDT, MKRUSDT, COMPUSDT
  layer1: 5 tokens - Layer 1 blockchain tokens
    ATOMUSDT, NEARUSDT, APTUSDT, SUIUSDT, INJUSDT

Total: 19 symbols


## 2. API Key Setup

**No API key required.** Binance Public API provides free access to historical data
through data.binance.vision.

The `ml4t-data` library handles rate limiting automatically.

In [3]:
print("Binance Public API requires no API key - data is freely available.")
print("Source: data.binance.vision")

Binance Public API requires no API key - data is freely available.
Source: data.binance.vision


## 3. Download Data

The download uses the `ml4t-data` library which handles:
- Rate limiting
- Data validation
- Consistent schema output

Two types of data are available:
- **OHLCV** (hourly): Price and volume for perpetual futures
- **Premium Index** (8-hourly): Basis between perpetual and spot prices

In [4]:
def download_crypto_ohlcv(
    dry_run: bool = False, force: bool = False, symbols: list[str] | None = None
):
    """Download crypto perpetual futures OHLCV from Binance.

    Args:
        dry_run: If True, show what would be downloaded without doing it
        force: If True, re-download even if data exists
        symbols: Specific symbols to download (default: all from config)
    """
    from utils import ML4T_DATA_PATH

    # Load config
    config = yaml.safe_load(config_path.read_text())
    crypto_config = config["crypto"]

    # Flatten symbols list
    if symbols is None:
        symbols = []
        for category_info in crypto_config["symbols"].values():
            symbols.extend(category_info["symbols"])

    output_dir = ML4T_DATA_PATH / "crypto" / "market"
    output_path = output_dir / "perps_1h.parquet"

    print("=== Crypto OHLCV Download ===")
    print(f"Symbols: {len(symbols)}")
    print("Frequency: 1h")
    print(f"Date range: {crypto_config['start']} to {crypto_config['end']}")
    print(f"Output: {output_path}")

    if dry_run:
        print("\n[DRY RUN] Would download:")
        for symbol in symbols:
            print(f"  {symbol}")
        return

    # Check existing
    if output_path.exists() and not force:
        existing = pl.read_parquet(output_path)
        print(f"\nData already exists ({len(existing):,} rows).")
        print("Use force=True to re-download.")
        return existing

    # Initialize provider
    from ml4t.data.providers import BinancePublicProvider

    provider = BinancePublicProvider(market="spot")

    # Download each symbol
    all_data = []
    print(f"\nDownloading {len(symbols)} symbols...")
    for symbol in symbols:
        print(f"  {symbol}...", end=" ", flush=True)
        try:
            df = provider.fetch_ohlcv(
                symbol=symbol,
                start=crypto_config["start"],
                end=crypto_config["end"],
                frequency="hourly",
            )
            df = df.with_columns(pl.lit(symbol).alias("symbol"))
            all_data.append(df)
            print(f"OK ({len(df):,} rows)")
        except Exception as e:
            print(f"ERROR: {e}")

    if not all_data:
        raise RuntimeError("No data downloaded!")

    # Combine and save
    output_dir.mkdir(parents=True, exist_ok=True)
    combined = pl.concat(all_data)
    combined.write_parquet(output_path)

    print("\n=== Complete ===")
    print(f"Total rows: {len(combined):,}")
    print(f"Symbols: {combined['symbol'].n_unique()}")
    print(f"Saved to: {output_path}")

    return combined


def download_crypto_premium(
    dry_run: bool = False, force: bool = False, symbols: list[str] | None = None
):
    """Download crypto premium index from Binance.

    Args:
        dry_run: If True, show what would be downloaded without doing it
        force: If True, re-download even if data exists
        symbols: Specific symbols to download (default: all from config)
    """
    from utils import ML4T_DATA_PATH

    # Load config
    config = yaml.safe_load(config_path.read_text())
    crypto_config = config["crypto"]

    # Flatten symbols list
    if symbols is None:
        symbols = []
        for category_info in crypto_config["symbols"].values():
            symbols.extend(category_info["symbols"])

    output_dir = ML4T_DATA_PATH / "crypto" / "market"
    output_path = output_dir / "premium_index_8h.parquet"

    print("=== Crypto Premium Index Download ===")
    print(f"Symbols: {len(symbols)}")
    print("Frequency: 8h (funding rate interval)")
    print(f"Date range: {crypto_config['start']} to {crypto_config['end']}")
    print(f"Output: {output_path}")

    if dry_run:
        print("\n[DRY RUN] Would download:")
        for symbol in symbols:
            print(f"  {symbol}")
        return

    # Check existing
    if output_path.exists() and not force:
        existing = pl.read_parquet(output_path)
        print(f"\nData already exists ({len(existing):,} rows).")
        print("Use force=True to re-download.")
        return existing

    # Initialize provider (futures market for premium index)
    from ml4t.data.providers import BinancePublicProvider

    provider = BinancePublicProvider(market="futures")

    print(f"\nDownloading premium index for {len(symbols)} symbols...")
    premium_data = provider.fetch_premium_index_multi(
        symbols=symbols,
        start=crypto_config["start"],
        end=crypto_config["end"],
        interval="8h",
    )

    # Save
    output_dir.mkdir(parents=True, exist_ok=True)
    premium_data.write_parquet(output_path)

    print("\n=== Complete ===")
    print(f"Total rows: {len(premium_data):,}")
    print(f"Symbols: {premium_data['symbol'].n_unique()}")
    print(f"Saved to: {output_path}")

    return premium_data

### Download OHLCV Data

In [5]:
# Uncomment to download OHLCV data
# download_crypto_ohlcv()

### Download Premium Index

In [6]:
# Uncomment to download premium index
# download_crypto_premium()

### Dry Run (Preview)

In [7]:
download_crypto_ohlcv(dry_run=True)

=== Crypto OHLCV Download ===
Symbols: 19
Frequency: 1h
Date range: 2020-01-01 to 2025-12-31
Output: data/crypto/market/perps_1h.parquet

[DRY RUN] Would download:
  BTCUSDT
  ETHUSDT
  BNBUSDT
  SOLUSDT
  XRPUSDT
  DOGEUSDT
  ADAUSDT
  AVAXUSDT
  DOTUSDT
  LINKUSDT
  UNIUSDT
  AAVEUSDT
  MKRUSDT
  COMPUSDT
  ATOMUSDT
  NEARUSDT
  APTUSDT
  SUIUSDT
  INJUSDT


## 4. Load and Explore

Once downloaded, use the loaders throughout the book:

In [8]:
from data import load_crypto_perps, load_crypto_premium

### Perpetual Futures OHLCV

In [9]:
# Load hourly OHLCV data
perps = load_crypto_perps()

print(f"Shape: {perps.shape}")
print(f"Symbols: {perps['symbol'].n_unique()}")
print(f"Date range: {perps['timestamp'].min()} to {perps['timestamp'].max()}")
print(f"Memory: {perps.estimated_size('mb'):.1f} MB")

Shape: (866484, 7)
Symbols: 19
Date range: 2020-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
Memory: 45.8 MB


In [10]:
# Schema
perps.schema

Schema([('timestamp', Datetime(time_unit='ms', time_zone='UTC')),
        ('open', Float64),
        ('high', Float64),
        ('low', Float64),
        ('close', Float64),
        ('volume', Float64),
        ('symbol', String)])

In [11]:
# Preview
perps.head(10)

timestamp,open,high,low,close,volume,symbol
"datetime[ms, UTC]",f64,f64,f64,f64,f64,str
2020-10-16 07:00:00 UTC,44.55,45.04,37.51,41.239,20786.7,"""AAVEUSDT"""
2020-10-16 08:00:00 UTC,41.248,43.324,40.673,42.532,18338.0,"""AAVEUSDT"""
2020-10-16 09:00:00 UTC,42.533,42.8,41.666,41.849,16102.2,"""AAVEUSDT"""
2020-10-16 10:00:00 UTC,41.854,42.021,40.914,41.432,15207.8,"""AAVEUSDT"""
2020-10-16 11:00:00 UTC,41.436,41.995,40.995,41.389,14631.2,"""AAVEUSDT"""
2020-10-16 12:00:00 UTC,41.399,41.822,41.048,41.681,15019.0,"""AAVEUSDT"""
2020-10-16 13:00:00 UTC,41.693,41.9,41.4,41.797,14674.4,"""AAVEUSDT"""
2020-10-16 14:00:00 UTC,41.794,42.345,41.65,41.75,15578.4,"""AAVEUSDT"""
2020-10-16 15:00:00 UTC,41.75,41.775,41.292,41.63,15112.3,"""AAVEUSDT"""


In [12]:
# Volume by symbol (USD notional)
volume_by_symbol = (
    perps.group_by("symbol")
    .agg(
        (pl.col("volume") * pl.col("close")).sum().alias("total_volume_usd"),
        pl.len().alias("n_observations"),
        pl.col("timestamp").min().alias("first_date"),
        pl.col("timestamp").max().alias("last_date"),
    )
    .sort("total_volume_usd", descending=True)
)
volume_by_symbol

symbol,total_volume_usd,n_observations,first_date,last_date
str,f64,u32,"datetime[ms, UTC]","datetime[ms, UTC]"
"""BTCUSDT""",2.9063e13,52608,2020-01-01 00:00:00 UTC,2025-12-31 23:00:00 UTC
"""ETHUSDT""",1.6658e13,52608,2020-01-01 00:00:00 UTC,2025-12-31 23:00:00 UTC
"""SOLUSDT""",3.7975e12,46313,2020-09-14 07:00:00 UTC,2025-12-31 23:00:00 UTC
"""XRPUSDT""",2.4606e12,52360,2020-01-06 08:00:00 UTC,2025-12-31 23:00:00 UTC
"""DOGEUSDT""",2.0613e12,48015,2020-07-10 09:00:00 UTC,2025-12-31 23:00:00 UTC
…,…,…,…,…
"""AAVEUSDT""",3.0024e11,45665,2020-10-16 07:00:00 UTC,2025-12-31 23:00:00 UTC
"""UNIUSDT""",2.9582e11,46337,2020-09-18 07:00:00 UTC,2025-12-31 23:00:00 UTC
"""INJUSDT""",1.6342e11,29590,2022-08-17 02:00:00 UTC,2025-12-31 23:00:00 UTC


### Premium Index

In [13]:
# Load 8-hourly premium index data
premium = load_crypto_premium()

print(f"Shape: {premium.shape}")
print(f"Symbols: {premium['symbol'].n_unique()}")
print(f"Date range: {premium['timestamp'].min()} to {premium['timestamp'].max()}")
print(f"Memory: {premium.estimated_size('mb'):.1f} MB")

Shape: (107839, 6)
Symbols: 19
Date range: 2020-01-01 00:00:00+00:00 to 2025-12-31 16:00:00+00:00
Memory: 4.9 MB


In [14]:
# Preview
premium.head(10)

timestamp,symbol,premium_index_open,premium_index_high,premium_index_low,premium_index_close
"datetime[ms, UTC]",str,f64,f64,f64,f64
2020-10-16 00:00:00 UTC,"""AAVEUSDT""",-0.001325,0.006456,-0.012625,0.0
2020-10-16 08:00:00 UTC,"""AAVEUSDT""",0.0,0.001896,-0.013317,0.0
2020-10-16 16:00:00 UTC,"""AAVEUSDT""",0.0,0.00383,-0.006755,0.0
2020-10-17 00:00:00 UTC,"""AAVEUSDT""",0.0,0.009527,-0.002972,-0.001254
2020-10-17 08:00:00 UTC,"""AAVEUSDT""",-0.003966,0.002304,-0.003966,0.0
2020-10-17 16:00:00 UTC,"""AAVEUSDT""",0.0,0.001807,-0.003141,0.0
2020-10-18 00:00:00 UTC,"""AAVEUSDT""",0.0,0.001466,-0.001297,0.0
2020-10-18 08:00:00 UTC,"""AAVEUSDT""",0.0,0.002181,-0.001459,0.0
2020-10-18 16:00:00 UTC,"""AAVEUSDT""",0.0,0.003928,-0.001558,0.0


In [15]:
# Premium statistics by symbol
# Premium index captures basis between perpetual and spot
premium_stats = (
    premium.group_by("symbol")
    .agg(
        pl.col("premium_index_close").mean().alias("mean_premium"),
        pl.col("premium_index_close").std().alias("std_premium"),
        pl.col("premium_index_close").min().alias("min_premium"),
        pl.col("premium_index_close").max().alias("max_premium"),
    )
    .sort("mean_premium", descending=True)
)
premium_stats

symbol,mean_premium,std_premium,min_premium,max_premium
str,f64,f64,f64,f64
"""MKRUSDT""",-0.000003,0.000744,-0.006005,0.008476
"""BNBUSDT""",-0.000032,0.000723,-0.011492,0.005062
"""LINKUSDT""",-0.000045,0.00068,-0.007393,0.0065638
"""XRPUSDT""",-0.000049,0.001109,-0.069842,0.006523
"""ETHUSDT""",-0.000055,0.000628,-0.002435,0.007106
…,…,…,…,…
"""COMPUSDT""",-0.000196,0.001009,-0.031857,0.006106
"""SUIUSDT""",-0.000219,0.000614,-0.006357,0.012701
"""INJUSDT""",-0.00025,0.000697,-0.00694,0.007107


## 5. Data Profile

Profiles document the dataset structure, statistics, and quality metrics.

In [16]:
from utils import ML4T_DATA_PATH

# Check for existing profiles
for dataset, filename in [
    ("OHLCV", "perps_1h_profile.json"),
    ("Premium", "premium_index_8h_profile.json"),
]:
    profile_path = ML4T_DATA_PATH / "crypto" / "market" / filename
    if profile_path.exists():
        profile = json.loads(profile_path.read_text())
        print(f"=== Crypto {dataset} Profile ===")
        rows = profile.get("total_rows", profile.get("rows"))
        cols = profile.get("total_columns", profile.get("columns"))
        if isinstance(cols, list):
            cols = len(cols)
        print(f"Rows: {rows:,}" if rows is not None else "Rows: unknown")
        print(f"Columns: {cols}")
        mem = profile.get("memory_mb")
        if mem is not None:
            print(f"Memory: {mem:.1f} MB")
        print()
    else:
        print(f"Profile not found: {profile_path}")
        print(f"Generate with: python generate_profiles.py --dataset crypto_{dataset.lower()}\n")

=== Crypto OHLCV Profile ===
Rows: 866,484
Columns: 7

=== Crypto Premium Profile ===
Rows: 107,839
Columns: 6



## 6. Loader Options

The loaders support filtering by symbols and date range:

In [17]:
# Specific symbols
btc_eth = load_crypto_perps(symbols=["BTCUSDT", "ETHUSDT"])
print(f"BTC + ETH only: {btc_eth.shape}")

BTC + ETH only: (105216, 7)


In [18]:
# Date range
recent = load_crypto_premium(start_date="2024-01-01")
print(f"Premium 2024+: {recent.shape}")

Premium 2024+: (41456, 6)


In [19]:
# Combined filters
filtered = load_crypto_perps(
    symbols=["BTCUSDT", "ETHUSDT", "SOLUSDT"], start_date="2023-01-01", end_date="2023-12-31"
)
print(f"3 symbols, 2023: {filtered.shape}")

3 symbols, 2023: (26280, 7)


## 7. Documentation

### Binance Public API
- [Binance Public Data](https://data.binance.vision/)
- [API Documentation](https://binance-docs.github.io/apidocs/spot/en/)

### Premium Index

The premium index measures the basis between perpetual futures and spot prices:

$$\text{Premium} = \frac{P_{perp} - P_{spot}}{P_{spot}}$$

Key properties:
- **Positive premium**: Perpetual trades at premium (bullish sentiment)
- **Negative premium**: Perpetual trades at discount (bearish sentiment)
- **Funding rate**: Derived from premium, settles every 8 hours

### Data Quality Notes
- Volume represents Binance exchange volume only
- BTC/ETH/major alts: Data from Jan 2020 (6 years history)
- Newer tokens (APT, SUI, INJ, ARB, OP): Data from listing date (2022-2023)
- MATICUSDT renamed to POLUSDT in Sept 2024 (data ends there)
- 8-hour intervals align with funding rate settlement times (00:00, 08:00, 16:00 UTC)

## 8. Updating Data

To update with the latest data, re-run the download:

```python
# Update OHLCV data
download_crypto_ohlcv()

# Update premium index
download_crypto_premium()

# Force full re-download
download_crypto_ohlcv(force=True)
download_crypto_premium(force=True)
```

**Tip**: Update the `end` date in `config.yaml` before re-downloading.

## Summary

| Item | Value |
|------|-------|
| Symbols | 20 perpetual futures (major, DeFi, L1) |
| Frequencies | 1h OHLCV, 8h premium index |
| Coverage | 2020-2025 (6 years for BTC/ETH) |
| Provider | Binance Public (free) |
| Config | `config.yaml` |
| Loaders | `load_crypto_perps()`, `load_crypto_premium()` |

**Use case**: Funding rate arbitrage strategy exploiting premium mean reversion.